# Treino RL self-hosted (`openpipe-art[megatron]`) — agente de reunião pt-BR

Mesmo pipeline do notebook Tinker (dataset sintético → rollout multi-turno → RULER → GRPO), trocando o backend pago pelo **stack Megatron aberto do ART** rodando em GPUs suas. É o "Tinker em casa": o Qwen3.6-27B e o Qwen3.6-35B-A3B são specs **validadas** no registry do Megatron do ART.

## ⚠️ Hardware — leia antes de rodar

A topologia mínima suportada é **2 GPUs**: uma para o trainer, outra para o vLLM dos rollouts (`trainer_gpu_ids=[0]`, `inference_gpu_ids=[1]` — é assim que os runs de referência do próprio ART operam).

| Modelo | Mínimo | Referência upstream |
|---|---|---|
| `Qwen/Qwen3.6-27B` (denso) | 2× GPU 80 GB (H100/A100-80) | — |
| `Qwen/Qwen3.6-35B-A3B` (MoE) | 2× H200 (o MoE usa expert parallelism) | `H200:2` |

**O Colab não serve para este notebook** (1 GPU por runtime). Use um pod com Jupyter: RunPod (template "Jupyter" com 2×H100 ≈ US$ 5,80/h ou 2×H200) ou Vast.ai, com imagem CUDA 12.8 (`nvidia/cuda:12.8.1-devel-ubuntu22.04`). A primeira célula verifica e te avisa.

**Custos de referência**: ~US$ 6–8/h de pod; o objetivo deste notebook com `NUM_STEPS=3` é **medir custo/step e tempo/step reais** para comparar com os ~US$ 3,33/step do Tinker antes de decidir a migração.

**Status honesto**: este notebook é um scaffold construído sobre os scripts dev oficiais do ART (`dev/yes-no-maybe-megatron.py`) — o stack Megatron é o mais pesado do ecossistema e a primeira execução costuma exigir iteração (instalação de 20–40 min, dependências CUDA sensíveis). Chaves: `OPENROUTER_API_KEY` (RULER), `OPENAI_API_KEY` (juiz), `GEMINI_API_KEY` (opcional), `HF_TOKEN` (recomendado). **Não precisa de TINKER_API_KEY.**

In [ ]:
import torch

n = torch.cuda.device_count()
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} — {p.total_memory / 1e9:.0f} GB")
assert n >= 2, (
    f"Este notebook requer >= 2 GPUs (trainer + inferência); encontrado: {n}. "
    "O Colab fornece só 1 — rode num pod RunPod/Vast com Jupyter e 2x H100/H200 "
    "(imagem CUDA 12.8).")
print("Topologia OK: trainer na GPU 0, vLLM (rollouts) na GPU 1.")

In [ ]:
# Instalação pesada (transformer-engine, flash-attn, megatron-core): 20-40 min.
# Mesmo commit pinado dos outros notebooks.
%pip uninstall -q -y openpipe-art
%pip install -q "openpipe-art[megatron] @ git+https://github.com/OpenPipe/ART.git@3492eb9a9ff1" openai langdetect python-dotenv

# Se o pip falhar em transformer-engine/apex: confirme imagem CUDA 12.8 + Python 3.11-3.13.

In [ ]:
import os

# Somente carrega (.env -> secrets do Colab/Jupyter); sem prompts.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except ImportError:
    print("python-dotenv ausente — rode a célula de instalação e reinicie o kernel.")
try:
    from google.colab import userdata
    for k in ("OPENAI_API_KEY", "OPENROUTER_API_KEY", "GEMINI_API_KEY", "HF_TOKEN"):
        if not os.environ.get(k):
            try:
                os.environ[k] = userdata.get(k) or ""
            except Exception:
                pass
except ImportError:
    pass

faltando = [k for k in ("OPENAI_API_KEY", "OPENROUTER_API_KEY") if not os.environ.get(k)]
assert not faltando, f"Chaves ausentes: {', '.join(faltando)} — preencha o .env"
if not os.environ.get("GEMINI_API_KEY"):
    print("GEMINI_API_KEY ausente — a expansão usará o cache em JSON ou será pulada.")
print("Chaves OK")

## 1. Dataset sintético pt-BR (idêntico ao notebook Tinker)

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Segment:
    idx: int
    speaker: str
    text: str

@dataclass
class MeetingScenario:
    id: str
    title: str
    date: str
    segments: list
    question: str
    expected: str = ""
    split: str = "train"

def _seg(rows):
    return [Segment(i, sp, tx) for i, (sp, tx) in enumerate(rows)]

SCENARIOS = [
    MeetingScenario(
        id="planning-q3", title="Planejamento Q3 — Produto", date="2026-06-10",
        segments=_seg([
            ("Ana", "Bom dia pessoal. Hoje precisamos fechar o escopo do Q3."),
            ("Bruno", "A prioridade número um continua sendo o app de desktop, certo?"),
            ("Ana", "Sim. O desktop tem que sair até o fim de julho, é compromisso com o cliente."),
            ("Carla", "Eu assumo a parte de captura de áudio do desktop então."),
            ("Bruno", "E eu fico com a integração do Teams. Fecho a POC até dia 20."),
            ("Ana", "Perfeito. A Carla entrega captura até 25 de julho e o Bruno a POC do Teams até 20 de junho."),
            ("Carla", "Combinado. Alguma dependência externa que eu precise saber?"),
            ("Ana", "Só o SDK de captura, mas isso já está aprovado pelo jurídico."),
        ]),
        question="Quais são as tarefas atribuídas e seus responsáveis e prazos?",
        expected="Carla: captura de áudio do desktop até 25/07. Bruno: POC de integração do Teams até 20/06.",
        split="train",
    ),
    MeetingScenario(
        id="incident-review", title="Revisão de Incidente — Fila de Transcrição", date="2026-06-12",
        segments=_seg([
            ("Diego", "A fila de transcrição travou ontem às 14h por duas horas."),
            ("Elena", "A causa foi o pico de reuniões simultâneas, o worker não escalou."),
            ("Diego", "Precisamos de auto-scaling na fila até o fim da semana."),
            ("Elena", "Eu configuro o auto-scaling. Sexta no máximo."),
            ("Diego", "E vamos adicionar um alerta de profundidade de fila também."),
            ("Fábio", "Eu crio o alerta de fila, posso entregar na quinta."),
            ("Diego", "Ótimo. Nenhum dado de cliente foi perdido, só houve atraso."),
        ]),
        question="O que causou o incidente e quais ações foram definidas para evitar recorrência?",
        expected="Causa: pico de reuniões simultâneas sem auto-scaling do worker. Ações: Elena configura auto-scaling até sexta; Fábio cria alerta de profundidade de fila até quinta.",
        split="train",
    ),
    MeetingScenario(
        id="sales-sync", title="Sync Comercial — Conta Northwind", date="2026-06-15",
        segments=_seg([
            ("Gabriela", "A Northwind quer expandir de 50 para 300 licenças."),
            ("Hugo", "Ótimo. Eles pediram algum requisito novo?"),
            ("Gabriela", "Sim, isolamento de dados por região e retenção de 90 dias."),
            ("Hugo", "Retenção configurável já temos. Isolamento por região é roadmap."),
            ("Gabriela", "Eles topam esperar o isolamento se tivermos data marcada."),
            ("Hugo", "Vou levantar o esforço de isolamento por região com o time de dados."),
            ("Gabriela", "Fecho a proposta de 300 licenças até quarta que vem."),
        ]),
        question="Quais requisitos o cliente pediu e qual é o próximo passo de cada pessoa?",
        expected="Requisitos: isolamento de dados por região e retenção de 90 dias. Próximos passos: Hugo levanta o esforço de isolamento por região; Gabriela fecha a proposta de 300 licenças até quarta.",
        split="train",
    ),
    MeetingScenario(
        id="design-review", title="Design Review — Bot do Teams", date="2026-06-18",
        segments=_seg([
            ("Iara", "O @TagBot precisa responder menção em canal e em DM."),
            ("João", "Em canal, só responde se for explicitamente mencionado, senão vira spam."),
            ("Iara", "Concordo. E em DM responde sempre."),
            ("João", "A latência alvo é abaixo de 3 segundos para a primeira resposta."),
            ("Iara", "Eu escrevo a spec da máquina de estados do bot até segunda."),
            ("João", "Eu prototipo o handler de menção até quarta."),
        ]),
        question="Quais decisões de comportamento do bot foram tomadas e quem entrega o quê?",
        expected="Decisões: responde em canal só quando mencionado, sempre em DM, latência alvo <3s. Iara escreve a spec até segunda; João prototipa o handler de menção até quarta.",
        split="train",
    ),
    MeetingScenario(
        id="retention-policy", title="Política de Retenção e Consentimento", date="2026-06-20",
        segments=_seg([
            ("Lara", "Todo áudio gravado precisa de consentimento explícito do organizador."),
            ("Marcos", "E a retenção padrão do áudio bruto?"),
            ("Lara", "Trinta dias para o áudio, um ano para a transcrição."),
            ("Marcos", "Cada tenant pode encurtar, mas não estender além disso."),
            ("Lara", "Exato. Eu documento a política até sexta."),
            ("Marcos", "Eu implemento o job de expiração de áudio até o fim do mês."),
        ]),
        question="Quais são as regras de consentimento e retenção definidas?",
        expected="Consentimento explícito do organizador para gravar áudio. Retenção: 30 dias para áudio bruto, 1 ano para transcrição; tenant pode encurtar mas não estender. Lara documenta até sexta; Marcos implementa o job de expiração até o fim do mês.",
        split="train",
    ),
    MeetingScenario(
        id="hiring-plan", title="Plano de Contratação — Time de IA", date="2026-06-22",
        segments=_seg([
            ("Núbia", "Precisamos de dois engenheiros de ML no Q3."),
            ("Otávio", "Um sênior para o pipeline de RL e um pleno para avaliação."),
            ("Núbia", "Eu abro as duas vagas até quinta."),
            ("Otávio", "Eu preparo o desafio técnico de RL até a semana que vem."),
            ("Núbia", "Meta é fechar as contratações até o fim de agosto."),
        ]),
        question="Quantas vagas, com quais perfis, e quais são as ações e prazos?",
        expected="Duas vagas: um ML sênior (pipeline de RL) e um pleno (avaliação). Núbia abre as vagas até quinta; Otávio prepara o desafio técnico até a semana seguinte; meta de fechar até fim de agosto.",
        split="val",
    ),
    MeetingScenario(
        id="pricing-debate", title="Debate de Precificação", date="2026-06-24",
        segments=_seg([
            ("Paula", "O plano por assento não está funcionando para clientes grandes."),
            ("Rui", "Proponho cobrança híbrida: assento mais consumo de minutos transcritos."),
            ("Paula", "Risco de assustar o cliente com conta variável."),
            ("Rui", "Colocamos um teto mensal para dar previsibilidade."),
            ("Paula", "Gostei. Eu modelo três cenários de receita até terça."),
            ("Rui", "Eu levanto o custo real de transcrição por minuto até segunda."),
        ]),
        question="Qual mudança de precificação foi proposta e quais tarefas ficaram definidas?",
        expected="Proposta: cobrança híbrida (assento + consumo de minutos transcritos) com teto mensal. Paula modela três cenários de receita até terça; Rui levanta o custo real por minuto até segunda.",
        split="val",
    ),
    MeetingScenario(
        id="empty-decisions", title="Bate-papo sem decisões", date="2026-06-25",
        segments=_seg([
            ("Sofia", "Só queria alinhar como todos estão, sem pauta fechada hoje."),
            ("Tiago", "Por mim tudo tranquilo, semana calma."),
            ("Sofia", "Ótimo. Então não temos nada a decidir, bom descanso a todos."),
        ]),
        question="Quais tarefas e responsáveis foram definidos nesta reunião?",
        expected="Nenhuma tarefa foi definida; a reunião não teve decisões.",
        split="val",
    ),
]

def train_scenarios():
    return [s for s in SCENARIOS if s.split == "train"]

def val_scenarios():
    return [s for s in SCENARIOS if s.split == "val"]

print(f"{len(train_scenarios())} cenários de treino, {len(val_scenarios())} de validação")

## 2. (Opcional) Expandir o dataset — usa o cache `scenarios_expanded.json` se existir

In [ ]:
import asyncio, json, random
from openai import AsyncOpenAI

N_EXTRA = 200  # rodada real
EXPANDED_PATH = "scenarios_expanded.json"

def _scenario_from_dict(d):
    return MeetingScenario(
        id=d["id"], title=d["title"], date=d["date"],
        segments=[Segment(**seg) for seg in d["segments"]],
        question=d["question"], expected=d.get("expected", ""),
        split=d.get("split", "train"),
    )

if os.path.exists(EXPANDED_PATH):
    extra = [_scenario_from_dict(d) for d in json.load(open(EXPANDED_PATH))]
    SCENARIOS.extend(extra)
    print(f"+{len(extra)} cenários carregados de {EXPANDED_PATH} -> "
          f"{len(train_scenarios())} treino / {len(val_scenarios())} val")
elif os.environ.get("GEMINI_API_KEY"):
    teacher = AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.environ["GEMINI_API_KEY"],
    )
    DOMAINS = [
        "planejamento de produto", "revisão de incidente de produção",
        "reunião comercial com cliente", "design review de software",
        "compliance e privacidade de dados", "contratação e time",
        "precificação e finanças", "retrospectiva de sprint",
    ]
    QTYPES = [
        "tarefas atribuídas com responsáveis e prazos",
        "decisões tomadas e seus motivos",
        "riscos ou bloqueios levantados",
        "próximos passos por pessoa",
    ]
    GEN_PROMPT = (
        'Gere uma reunião de trabalho FICTÍCIA e realista em português do Brasil.\n'
        'Domínio: {domain}\nA reunião {control} conter tarefas/decisões concretas.\n'
        'Número de segmentos de fala: entre 25 e 60 (segmentos curtos, 1-3 frases, '
        'como saída real de diarização; inclua digressões e small talk).\n'
        'Depois formule UMA pergunta sobre: {qtype}. Por fim escreva a resposta ideal '
        '(objetiva, fiel à transcrição; se não houver tarefas/decisões, diga isso).\n'
        'Responda APENAS com JSON válido: {{"title": "...", "date": "AAAA-MM-DD", '
        '"segments": [{{"speaker": "Nome", "text": "..."}}], "question": "...", "expected": "..."}}'
    )

    async def gen_one(i):
        rng = random.Random(i)
        is_control = rng.random() < 0.15
        prompt = GEN_PROMPT.format(
            domain=rng.choice(DOMAINS),
            control="NÃO deve" if is_control else "deve",
            qtype=rng.choice(QTYPES),
        )
        try:
            resp = await teacher.chat.completions.create(
                model="gemini-2.5-flash-lite",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                max_tokens=4096, temperature=1.0,
            )
            d = json.loads(resp.choices[0].message.content or "")
            return MeetingScenario(
                id=f"gen-{i:04d}", title=d["title"], date=d["date"],
                segments=_seg([(s["speaker"], s["text"]) for s in d["segments"]]),
                question=d["question"], expected=d["expected"],
                split="train" if i % 10 else "val",  # ~10% para validação
            )
        except Exception:
            return None

    extra = [s for s in await asyncio.gather(*(gen_one(i) for i in range(N_EXTRA))) if s]
    SCENARIOS.extend(extra)
    from dataclasses import asdict
    with open(EXPANDED_PATH, "w") as f:
        json.dump([asdict(s) for s in extra], f, ensure_ascii=False, indent=1)
    print(f"Cenários salvos em {EXPANDED_PATH} (reutilizados nas próximas execuções)")
    print(f"+{len(extra)} cenários gerados -> {len(train_scenarios())} treino / {len(val_scenarios())} val")
    print("Inspecione algumas amostras antes de treinar:")
    if extra:
        s = extra[0]
        print(f"  [{s.id}] {s.title} — {len(s.segments)} segmentos\n  Q: {s.question}\n  E: {s.expected[:120]}")
else:
    print("Sem GEMINI_API_KEY nem cache — usando só os 8 cenários base (smoke test).")

## 3. Ferramentas do agente e helpers (idêntico)

In [ ]:
import re

TOOLS = [
    {"type": "function", "function": {
        "name": "search_transcript",
        "description": "Busca segmentos da transcrição que contenham as palavras-chave.",
        "parameters": {"type": "object", "properties": {
            "keywords": {"type": "array", "items": {"type": "string"}}},
            "required": ["keywords"]}}},
    {"type": "function", "function": {
        "name": "read_segment",
        "description": "Lê o texto completo de um segmento pelo seu índice.",
        "parameters": {"type": "object", "properties": {
            "segment_idx": {"type": "integer"}}, "required": ["segment_idx"]}}},
    {"type": "function", "function": {
        "name": "return_final_answer",
        "description": "Devolve a resposta final ao usuário e encerra a tarefa.",
        "parameters": {"type": "object", "properties": {
            "answer": {"type": "string"},
            "segment_refs": {"type": "array", "items": {"type": "integer"}}},
            "required": ["answer"]}}},
]

def _search(scenario, keywords):
    # O modelo às vezes manda keywords como string ("a, b, c") em vez de lista;
    # sem normalizar, o for itera caractere a caractere e casa com tudo.
    if isinstance(keywords, str):
        keywords = [k.strip() for k in keywords.split(",")]
    kws = [k.lower() for k in keywords if isinstance(k, str) and k.strip()]
    hits = [{"segment_idx": s.idx, "speaker": s.speaker, "snippet": s.text[:80]}
            for s in scenario.segments
            if any(k in s.text.lower() or k in s.speaker.lower() for k in kws)]
    return json.dumps({"results": hits[:10]}, ensure_ascii=False)

def _read(scenario, idx):
    for s in scenario.segments:
        if s.idx == idx:
            return json.dumps({"segment_idx": s.idx, "speaker": s.speaker, "text": s.text}, ensure_ascii=False)
    return json.dumps({"error": f"segmento {idx} não existe"}, ensure_ascii=False)

def _strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def _choice_as_context(choice):
    msg = choice.message
    out = {"role": "assistant", "content": _strip_think(msg.content or "")}
    if msg.tool_calls:
        out["tool_calls"] = [{"id": tc.id, "type": "function",
                              "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                             for tc in msg.tool_calls]
    return out

SYSTEM_PROMPT = (
    "Você é um assistente de reuniões em português do Brasil. Sua tarefa é "
    "responder à pergunta do usuário consultando a transcrição de uma reunião "
    "por meio das ferramentas disponíveis. Use search_transcript para "
    "localizar segmentos relevantes e read_segment para lê-los na íntegra. "
    "Responda SEMPRE em português, de forma objetiva e fiel à transcrição — "
    "não invente tarefas, decisões ou nomes que não estejam nos segmentos. "
    "Quando tiver a resposta, chame return_final_answer."
)

## 4. Modelo base e registro no ART (`MegatronBackend`)

A única célula estruturalmente diferente do notebook Tinker. `trainer_gpu_ids`/`inference_gpu_ids` dividem as GPUs; `gpu_memory_utilization` e `max_model_len` controlam o vLLM dos rollouts. Para o **35B-A3B** troque o `BASE_MODEL` (exige 2×H200).

In [ ]:
import art
from art.megatron import MegatronBackend

BASE_MODEL = "Qwen/Qwen3.6-27B"      # denso, 2x GPU 80GB
# BASE_MODEL = "Qwen/Qwen3.6-35B-A3B"  # MoE: use 2x H200

GPU_USD_HOUR = 5.80   # <-- EDITE: custo/hora do seu pod (para o relatório de custo)

internal_config = art.dev.InternalModelConfig(
    engine_args=art.dev.EngineArgs(
        gpu_memory_utilization=0.75,
        max_model_len=16384,
    ),
    trainer_gpu_ids=[0],
    inference_gpu_ids=[1],
)

backend = MegatronBackend()
model = art.TrainableModel(
    name="meeting-agent-ptbr-mgt-001",
    project="tag-ai-meeting-agent",
    base_model=BASE_MODEL,
    _internal_config=internal_config,
)
await model.register(backend)
print(f"Modelo registrado ({BASE_MODEL}); step atual: {await model.get_step()}")

## 5. Rollout (idêntico — com workaround multi-turn e resposta forçada)

In [ ]:
from art.trajectories import History

try:
    from langdetect import detect
except ImportError:
    detect = None

MAX_TURNS = 10

async def rollout(model, scenario):
    client = model.openai_client()
    # gpt-5.x na API da OpenAI rejeita max_tokens; modelos treináveis
    # (proxy do Tinker) usam max_tokens normalmente.
    token_kwargs = ({"max_tokens": 1024} if getattr(model, "trainable", False)
                    else {"max_completion_tokens": 1024})
    trajectory = art.Trajectory(
        messages_and_choices=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Reunião: {scenario.title} ({scenario.date}). "
                f"A transcrição tem {len(scenario.segments)} segmentos.\n\n"
                f"Pergunta: {scenario.question}")},
        ],
        metadata={"scenario_id": scenario.id, "split": scenario.split},
    )
    final_answer = None

    for _ in range(MAX_TURNS):
        completion = await client.chat.completions.create(
            # O nome de inferência é resolvido pelo backend (artifact do
            # W&B no serverless, alias name@step no Tinker/local) — nunca
            # passe model.name direto.
            model=model.get_inference_name(),
            messages=trajectory.messages(),
            tools=TOOLS, **token_kwargs,
        )
        choice = completion.choices[0]
        trajectory.messages_and_choices.append(choice)
        tool_calls = choice.message.tool_calls or []
        if not tool_calls:
            break
        for tc in tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            try:
                if tc.function.name == "search_transcript":
                    result = _search(scenario, args.get("keywords", []))
                elif tc.function.name == "read_segment":
                    idx = args.get("segment_idx", -1)
                    if isinstance(idx, list):  # o modelo às vezes manda lista
                        idx = idx[0] if idx else -1
                    result = _read(scenario, int(idx))
                elif tc.function.name == "return_final_answer":
                    final_answer = args.get("answer", "")
                    result = json.dumps({"ok": True}, ensure_ascii=False)
                else:
                    result = json.dumps({"error": "ferramenta desconhecida"}, ensure_ascii=False)
            except Exception as e:
                # Argumento malformado vira feedback para o modelo, nunca crash
                # do rollout (uma exceção derruba o gather do step inteiro).
                result = json.dumps({"error": f"argumentos inválidos: {e}"}, ensure_ascii=False)
            trajectory.messages_and_choices.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result})
        if final_answer is not None:
            break

    # Último recurso: estourou os turnos sem responder? Força uma resposta com
    # o que já foi lido — resposta parcial pontua; vazia é zero garantido.
    # (tool_choice é honrado pela API da OpenAI e ignorado sem erro pelo proxy
    # do Tinker; a instrução no user message cobre os dois casos.)
    if final_answer is None:
        trajectory.messages_and_choices.append({
            "role": "user",
            "content": ("Os turnos de busca acabaram. Responda AGORA chamando a "
                        "ferramenta return_final_answer com a melhor resposta "
                        "possível com base no que você já leu da transcrição."),
        })
        completion = await client.chat.completions.create(
            model=model.get_inference_name(),
            messages=trajectory.messages(),
            tools=TOOLS,
            tool_choice={"type": "function",
                         "function": {"name": "return_final_answer"}},
            **token_kwargs,
        )
        choice = completion.choices[0]
        trajectory.messages_and_choices.append(choice)
        for tc in choice.message.tool_calls or []:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if tc.function.name == "return_final_answer":
                final_answer = args.get("answer", "")
            trajectory.messages_and_choices.append(
                {"role": "tool", "tool_call_id": tc.id,
                 "content": json.dumps({"ok": True}, ensure_ascii=False)})

    trajectory.metrics["answered"] = 1.0 if final_answer else 0.0
    if detect and final_answer:
        try:
            trajectory.metrics["is_pt"] = 1.0 if detect(final_answer) == "pt" else 0.0
        except Exception:
            trajectory.metrics["is_pt"] = 0.0
    trajectory.metadata["final_answer"] = final_answer or ""
    return trajectory

# Workaround multi-turn Qwen 3.x: cada turno treina exatamente uma vez.
# IMPORTANTE: aplicar só DEPOIS do RULER — ele não aceita additional_histories
# (ValueError em art/rewards/ruler.py); o treino aceita.
def split_multiturn_for_training(trajectory):
    msgs = trajectory.messages_and_choices
    a_pos = [i for i, m in enumerate(msgs) if not isinstance(m, dict)]
    if len(a_pos) > 1:
        def _hist(pos):
            ctx = [m if isinstance(m, dict) else _choice_as_context(m) for m in msgs[:pos]]
            return ctx + [msgs[pos]]
        trajectory.additional_histories = [
            History(messages_and_choices=_hist(p)) for p in a_pos[1:]]
        trajectory.messages_and_choices = msgs[: a_pos[0] + 1]
    return trajectory

print("rollout definido")

## 6. Juiz de correção (idêntico)

In [ ]:
judge_client = AsyncOpenAI()  # OPENAI_API_KEY
CORRECTNESS_JUDGE = "gpt-5.4"

import re as _re

async def judge_correctness(scenario, answer):
    """Crédito parcial: fração (0-1) dos pontos da referência capturados."""
    if not scenario.expected:
        return 0.0
    prompt = (
        "Você avalia a resposta de um assistente de reuniões contra uma "
        "referência. Considere os pontos de informação da referência (tarefas, "
        "responsáveis, prazos, decisões, riscos). Calcule a fração deles "
        "corretamente capturada pela resposta, mesmo com outras palavras. "
        "Desconte por informações inventadas que contradigam a referência. "
        "Responda APENAS com um número inteiro de 0 a 100.\n\n"
        f"Pergunta: {scenario.question}\nReferência: {scenario.expected}\n"
        f"Resposta do agente: {answer}\n")
    resp = await judge_client.chat.completions.create(
        model=CORRECTNESS_JUDGE,
        messages=[{"role": "user", "content": prompt}],
        # gpt-5.x: max_tokens não é aceito; orçamento inclui raciocínio.
        max_completion_tokens=512)
    text = (resp.choices[0].message.content or "").strip()
    m = _re.search(r"\d{1,3}", text)
    return min(100, int(m.group())) / 100.0 if m else 0.0

async def validate(model):
    scenarios = val_scenarios()
    trajs = await asyncio.gather(*(rollout(model, s) for s in scenarios))
    scores = await asyncio.gather(*(
        judge_correctness(s, t.metadata.get("final_answer", ""))
        for s, t in zip(scenarios, trajs)))
    acc = sum(scores) / len(scores) if scores else 0.0
    ans = (sum(t.metrics.get("answered", 0.0) for t in trajs) / len(trajs)) if trajs else 0.0
    # pt medido só entre as respondidas — senão "não respondeu" contamina o idioma.
    pt_vals = [t.metrics["is_pt"] for t in trajs if "is_pt" in t.metrics]
    pt = (sum(pt_vals) / len(pt_vals)) if pt_vals else float("nan")
    print(f"  [validação] correção média (parcial): {acc:.2%} ({len(scores)} cenários) | "
          f"respondeu: {ans:.0%} | pt (entre respondidas): {pt:.0%}")
    return acc

## 7. Treino: gather → RULER → GRPO — **experimento de custo**

`NUM_STEPS=3` de propósito: o objetivo desta primeira execução é medir **tempo/step e custo/step reais** e comparar com o Tinker (~US$ 3,33/step). Cada step imprime a duração e o custo estimado pelo `GPU_USD_HOUR`. Se os números fecharem, suba para 25 e rode a rodada real — o loop retoma do step salvo.

In [ ]:
from art.rewards import ruler_score_group

# Juiz via OpenRouter (mesmo modelo): pool de capacidade próprio, sem o
# TPM de 200k da org na OpenAI. Requer OPENROUTER_API_KEY.
RULER_JUDGE_MODEL = "openrouter/openai/gpt-5.4-mini"
# Retries com backoff: o burst de ~44 juizes simultâneos (7-8k tokens cada)
# estoura o TPM da OpenAI; sem retry o erro é engolido e o reward zera.
RULER_LITELLM_PARAMS = {"num_retries": 6}
# Rubrica custom: alinha a recompensa com a métrica final (completude fiel).
RULER_RUBRIC = """
    - O objetivo do agente é responder à pergunta com fidelidade TOTAL à transcrição.
    - COMPLETUDE é o critério dominante: uma resposta que cobre todos os itens
      presentes na transcrição (todas as tarefas, responsáveis, prazos, decisões
      ou riscos pedidos) deve pontuar muito acima de uma que omite itens.
      Omissão é tão grave quanto invenção.
    - Penalize fortemente informação inventada (nomes, prazos ou decisões que
      não estão na transcrição).
    - Se a transcrição não contém a informação pedida, a melhor resposta diz
      isso explicitamente, sem inventar.
    - Respostas devem estar em português do Brasil.
    - Dê crédito parcial proporcional à fração dos itens corretamente cobertos;
      diferenças pequenas de qualidade devem gerar diferenças pequenas de nota.
"""
NUM_STEPS = 3   # experimento de custo/tempo; suba para 25 na rodada real
ROLLOUTS_PER_GROUP = 6   # RULER ranqueia melhor com 4-8 trajetórias por grupo
GROUPS_PER_STEP = 24     # subamostra de cenários por step (custo x diversidade)
LEARNING_RATE = 1.2e-5

import random
import time

for step in range(await model.get_step(), NUM_STEPS):
    print(f"\n=== Step {step} ===")
    _t_step = time.monotonic()
    pool = train_scenarios()
    scs = random.Random(step).sample(pool, min(GROUPS_PER_STEP, len(pool)))
    groups = await art.gather_trajectory_groups(
        (art.TrajectoryGroup(rollout(model, sc) for _ in range(ROLLOUTS_PER_GROUP))
         for sc in scs),
        pbar_desc="gather",
        # Tolera falhas pontuais (rede etc.) descartando o grupo afetado
        # em vez de matar o step; default 0 aborta na 1ª exceção.
        max_exceptions=12,
        after_each=lambda g: ruler_score_group(
            g, RULER_JUDGE_MODEL,
            extra_litellm_params=RULER_LITELLM_PARAMS,
            rubric=RULER_RUBRIC,
            swallow_exceptions=True),
    )
    # Grupo cujo juiz falhou vira None (swallow_exceptions) — descarta para
    # não treinar com rewards zerados.
    dropped = sum(g is None for g in groups)
    groups = [g for g in groups if g is not None]
    if dropped:
        print(f"⚠️ {dropped} grupo(s) descartado(s) por falha no juiz RULER")
    assert groups, "todos os grupos falharam no RULER — juiz/rate limit?"
    # Split multi-turn só agora: o RULER (after_each acima) já pontuou os grupos.
    for g in groups:
        for t in g.trajectories:
            split_multiturn_for_training(t)
    # No ART >=0.5 o treino é disparado pelo backend (model.train foi removido).
    result = await backend.train(
        model, groups, learning_rate=LEARNING_RATE,
        # Requerido pelo trainer Megatron (empacotamento de sequências).
        packed_sequence_length=16384,
        # Salva também o estado do otimizador (retomada + export garantidos).
        save_checkpoint=True,
    )
    await model.log(groups, metrics=result.metrics, step=result.step, split="train")
    _dt = time.monotonic() - _t_step
    print(f"Step {result.step} concluído em {_dt/60:.1f} min "
          f"(custo ≈ US$ {_dt/3600 * GPU_USD_HOUR:.2f} @ US$ {GPU_USD_HOUR}/h)")

print("\nTreino finalizado.")

## 8. Validação rápida (mesmo protocolo)

In [ ]:
final = await validate(model)
print(f"\nCorreção média no split de validação: {final:.2%}")

## 9. Backup do checkpoint no Hugging Face

Aqui os checkpoints LoRA ficam **no disco do pod** (`.art/…/checkpoints/`) — que morre com o pod. Faça backup antes de desligar.

In [ ]:
import glob
from art.utils.output_dirs import get_model_dir
from huggingface_hub import HfApi

HF_USER = "seu-usuario"   # <-- EDITE
REPO = f"{HF_USER}/meeting-agent-ptbr-milestones"

ckpts = sorted(glob.glob(f"{get_model_dir(model)}/checkpoints/*"))
assert ckpts, "Nenhum checkpoint — rode o treino primeiro."
last = ckpts[-1]
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(repo_id=REPO, folder_path=last,
                  path_in_repo=f"meeting-agent-ptbr-mgt-001/{os.path.basename(last)}")
print(f"Checkpoint {os.path.basename(last)} salvo em https://huggingface.co/{REPO}")